# Analysis of Stiffness Matrix

Below code analyses the stiffness matrix obtained from the respective Kglobal.dat file generated during the solving phase.
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.



To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [7]:
import os
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

os.chdir(r"C:\Calculix\ccx_2.23_wsl\Stiffness matrices\Scenario1 Full K")

sti_file = "K_baselinerefined.mtx"

def read_stiffness_matrix(sti_file):
    rows, cols, data = [], [], []

    with open(sti_file, 'r') as f:
        for line in f:
            parts = line.split()
            if len(parts) != 3:
                continue

            i = int(parts[0]) - 1
            j = int(parts[1]) - 1
            val = float(parts[2])

            rows.append(i)
            cols.append(j)
            data.append(val)

    n = max(max(rows), max(cols)) + 1
    K = sp.coo_matrix((data, (rows, cols)), shape=(n, n))

    # enforce symmetry
    K = K + K.T - sp.diags(K.diagonal())

    return K.tocsr()


K = read_stiffness_matrix(sti_file)

print(f"Matrix size: {K.shape}")
print(f"Nonzeros: {K.nnz}")

# --- eigenvalues (robust) ---
eigvals = spla.eigsh(K, k=10, sigma=0, which='LM', return_eigenvectors=False)

print("\nFirst 10 eigenvalues:")
print(eigvals)

# --- trace ---
trace_K = K.diagonal().sum()
print("\nTrace:", trace_K)

# --- condition number ---
lambda_max = spla.eigsh(K, k=1, which='LM', return_eigenvectors=False)[0]
lambda_min = spla.eigsh(K, k=1, sigma=0, which='LM', return_eigenvectors=False)[0]

cond_number = lambda_max / lambda_min
print("\nCondition number:", cond_number)

Matrix size: (3060, 3060)
Nonzeros: 338938

First 10 eigenvalues:
[2.54967251e+03 2.54967253e+03 9.75820727e+04 9.75820727e+04
 4.77316897e+05 7.34915564e+05 7.34915564e+05 2.49547910e+06
 2.66918826e+06 2.66918826e+06]

Trace: 48838096270054.84

Condition number: 53322211.366486594
